Analisis sastra otomatis untuk bab buku (BookSum) - Kelompok 2A INT.
Tugas yang dikerjakan adalah conditional text generation dari `chapter` ke `summary_analysis`.
Notebook ini hanya memanggil skrip di folder `src/` pada repo
https://github.com/RayDitya12/booksum-literary-analysis, jadi perubahan logika sebaiknya dibuat di sana,
bukan di dalam cell ini.
Jalankan cell dari atas ke bawah. Kalau sesi terputus, lanjutkan dari cell pemulihan di paling bawah.

1. Siapkan runtime dan folder proyek

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

Baris `nvidia-smi` harus menampilkan Tesla T4. Kalau tidak, pilih Runtime > Change runtime type >
T4 GPU, lalu jalankan ulang cell di atas. Tanpa GPU, pengaturan `fp16` di `src/config.py` akan gagal.

Cell di bawah mengklik repo kalau belum ada, lalu mengambil perubahan terbaru dengan `git pull`. Jangan
menggantinya dengan `rm -rf` lalu clone ulang: model hasil training tersimpan di folder yang sama dan akan
ikut terhapus.

In [ ]:
%cd /content
!git clone -q https://github.com/RayDitya12/booksum-literary-analysis.git
%cd /content/booksum-literary-analysis
!git pull -q origin main
!pip install -q -r requirements.txt

2. Periksa kondisi dataset

In [ ]:
!python src/verify.py

Baris terakhir harus berbunyi `CHECKS ... = PASS` untuk keempat pemeriksaan. Angka yang kami dapat:
train 9.600 menjadi 3.882 baris dari 110 buku, validation 1.484 menjadi 884, test 1.431 menjadi 479.

3. Bangun dataset hasil filter dan kolom chapter_selected

In [ ]:
!python src/data.py

Tahap ini hanya memakai CPU. Hasilnya tersimpan di `outputs/dataset`.

4. Ukur panjang teks dalam token

In [ ]:
!python src/eda.py --limit 300

Tahap ini yang menentukan nilai `max_src_tokens` di `src/config.py`, jadi jalankan sebelum training.
Angka acuan kami pada 300 baris train:

| input | token mean | cakupan 1024 | cakupan 8192 |
|---|---|---|---|
| `chapter` apa adanya | 4.170 | 7,7% | 92,0% |
| `chapter_selected` 500 kata | 699 | 99,0% | 100% |

Kalau cakupan Model 2 turun jauh di bawah 99%, jangan lanjut: nilai `HYBRID_MAX_WORDS` perlu ditinjau lagi.
Untuk target, 10,3% baris `summary_analysis` melebihi 1024 token. Itu batas arsitektur model, bukan hal yang
bisa dinaikkan lewat pengaturan training.

5. Uji kecil lebih dulu sebelum training penuh

In [ ]:
!python src/train.py --arm m1_bart_direct --limit 30 --epochs 0.05

In [ ]:
!python src/evaluate.py --arm m1_bart_direct --limit 10

Dua cell ini biasanya menemukan error lebih dulu karena menyentuh API library yang berubah antar versi.
Kalau keduanya gagal, perbaiki dulu sebelum menjalankan training penuh.

6. Training tiga varian model, satu varian per cell

In [ ]:
!pkill -f 'nvidia-smi --query-gpu' 2>/dev/null
!nohup nvidia-smi --query-gpu=memory.used --format=csv -l 30 > /content/drive/MyDrive/vram_m1.csv &
!python src/train.py --arm m1_bart_direct

In [ ]:
!pkill -f 'nvidia-smi --query-gpu' 2>/dev/null
!nohup nvidia-smi --query-gpu=memory.used --format=csv -l 30 > /content/drive/MyDrive/vram_m2.csv &
!python src/train.py --arm m2_bart_hybrid

In [ ]:
!pkill -f 'nvidia-smi --query-gpu' 2>/dev/null
!nohup nvidia-smi --query-gpu=memory.used --format=csv -l 30 > /content/drive/MyDrive/vram_m3.csv &
!python src/train.py --arm m3_led_long

Urutan dari yang paling ringan disengaja, supaya dua varian pertama tetap selesai kalau sesi habis di tengah jalan.
Waktu latih tersimpan di `outputs/<varian>/train_meta.json` dan angka itu yang dipakai di laporan.
Pemakaian VRAM tiap varian ditulis ke `vram_m1.csv`, `vram_m2.csv`, `vram_m3.csv` di Drive; pakai nilai tertinggi tiap file sebagai bukti kebutuhan memori per varian.
Kalau `m3_led_long` berhenti dengan `CUDA out of memory`, turunkan `max_src_tokens` varian itu ke 6144 di
`src/config.py` lalu jalankan ulang, dan catat penurunannya.
Sesi Colab tidak bisa melanjutkan training dari tengah epoch, jadi kalau satu varian tidak selesai dalam satu sesi,
kurangi `EPOCHS` dan laporkan jumlah epoch yang benar-benar dijalankan.

7. Evaluasi dan tabel perbandingan

In [ ]:
!python src/evaluate.py --arm m1_bart_direct
!python src/evaluate.py --arm m2_bart_hybrid
!python src/evaluate.py --arm m3_led_long
!cat outputs/leaderboard.jsonl

Satu kali pemanggilan sudah memuat greedy dan beam search, jadi tidak perlu dipanggil dua kali per varian.
`leaderboard.jsonl` adalah tabel yang dipakai di bab hasil.

8. Demo

In [ ]:
!python src/demo.py --arm m3_led_long

Cell ini tidak pernah selesai karena servernya berjalan terus, dan URL `gradio.live` muncul di output cell.
Hentikan cell untuk melepasnya. Ganti `--arm` untuk membandingkan varian.

Simpan hasil ke Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/booksum-outputs
!cp -r outputs/. /content/drive/MyDrive/booksum-outputs/
!ls /content/drive/MyDrive/booksum-outputs

Jalankan cell ini sesudah setiap tahap selesai, bukan hanya di akhir. Isi `/content` ikut hilang saat sesi terputus,
sedangkan `outputs/dataset` dan `outputs/<varian>` adalah yang membuat pekerjaan sebelumnya tidak perlu diulang.

Setelah sesi baru, pulihkan dengan cell berikut:

In [ ]:
!mkdir -p /content/booksum-literary-analysis/outputs
!cp -r /content/drive/MyDrive/booksum-outputs/. /content/booksum-literary-analysis/outputs/
%cd /content/booksum-literary-analysis